# Notebook 02 — Experiment with Calibration

Fit the minimal model, apply isotonic calibration, and compare calibration
curves and proper scoring rules (Brier, log-loss).

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.datasets import make_classification

from phase01_lightgbm_baseline import LightGBMWrapper

X, y = make_classification(
    n_samples=5000,
    n_features=10,
    n_informative=5,
    weights=[0.965, 0.035],
    flip_y=0.02,
    random_state=7,
)
X_tr, X_tmp, y_tr, y_tmp = train_test_split(X, y, test_size=0.4, stratify=y, random_state=7)
X_val, X_te, y_val, y_te = train_test_split(X_tmp, y_tmp, test_size=0.5, stratify=y_tmp, random_state=7)
print(y_te.mean())

In [ ]:
model = LightGBMWrapper()
model.train(X_tr, y_tr, X_val, y_val)

raw_test = model.predict(X_te)
model.calibrate(model.predict(X_val), y_val)
cal_test = model.calibrator.predict(raw_test)

from sklearn.metrics import brier_score_loss, log_loss

print(f"raw  Brier: {brier_score_loss(y_te, raw_test):.4f}")
print(f"cal  Brier: {brier_score_loss(y_te, cal_test):.4f}")
print(f"raw  log-loss: {log_loss(y_te, np.clip(raw_test, 1e-9, 1 - 1e-9)):.4f}")
print(f"cal  log-loss: {log_loss(y_te, np.clip(cal_test, 1e-9, 1 - 1e-9)):.4f}")

In [ ]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from sklearn.calibration import calibration_curve

frac_raw, mean_raw = calibration_curve(y_te, raw_test, n_bins=10, strategy="quantile")
frac_cal, mean_cal = calibration_curve(y_te, cal_test, n_bins=10, strategy="quantile")

fig, ax = plt.subplots(figsize=(5, 5))
ax.plot([0, 1], [0, 1], ls="--", color="grey", label="perfect")
ax.plot(mean_raw, frac_raw, "o-", label="raw")
ax.plot(mean_cal, frac_cal, "s-", label="isotonic")
ax.set(xlabel="Predicted probability", ylabel="Observed fraud rate", title="Reliability diagram")
ax.legend()
fig.tight_layout()
fig.savefig("nb02_calibration.png", dpi=120)
print("saved nb02_calibration.png")

Isotonic calibration pushes scores toward the diagonal, lowering the Brier score.